# HDC PM2.5 API export

Notebook นี้ดึงข้อมูล `s_pm25_1_in_week` จาก OpenData MOPH API ทุกจังหวัด แล้ว clean เป็น long format ด้วยคอลัมน์ปลายทาง:

`no, province_code, province_name, county, year, week, month, typediag_id, typediag, icd10, Typediag_name, diagnosis, case`


In [6]:
from pathlib import Path
import datetime as dt
import os
import re
import time

import pandas as pd
import requests

API_URL = os.getenv("HDC_API_URL", "https://opendata.moph.go.th/api/report_data")
TABLE_NAME = os.getenv("HDC_TABLE_NAME", "s_pm25_1_in_week")
YEAR = os.getenv("HDC_YEAR_THAI", "2569")
REQUEST_TIMEOUT_SEC = int(os.getenv("HDC_REQUEST_TIMEOUT_SEC", "60"))
RETRY_COUNT = int(os.getenv("HDC_RETRY_COUNT", "2"))
SLEEP_BETWEEN_REQUESTS_SEC = float(os.getenv("HDC_SLEEP_BETWEEN_REQUESTS_SEC", "0.25"))
FAIL_ON_PROVINCE_ERROR = os.getenv("HDC_FAIL_ON_PROVINCE_ERROR", "true").lower() not in {"0", "false", "no"}

CWD = Path.cwd()
if (CWD / "airflow" / "dags" / "scripts").exists():
    OUTPUT_DIR = CWD / "airflow" / "dags" / "scripts"
elif (CWD / "scripts").exists():
    OUTPUT_DIR = CWD / "scripts"
else:
    OUTPUT_DIR = CWD
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LONG_OUTPUT_FILE = OUTPUT_DIR / f"hdc_merged_long_{YEAR}.csv"
SUMMARY_OUTPUT_FILE = OUTPUT_DIR / f"hdc_report_summary_{YEAR}.csv"

FINAL_COLUMNS = [
    "no",
    "province_code",
    "province_name",
    "county",
    "year",
    "week",
    "month",
    "typediag_id",
    "typediag",
    "icd10",
    "Typediag_name",
    "diagnosis",
    "case",
]

print("Output dir:", OUTPUT_DIR.resolve())


Output dir: /Users/champ/Documents/dev/envocc-dashboard-pm/airflow/dags/scripts


In [7]:
PROVINCE_ID_MAPPING = {
    "กรุงเทพมหานคร": "10",
    "สมุทรปราการ": "11",
    "นนทบุรี": "12",
    "ปทุมธานี": "13",
    "พระนครศรีอยุธยา": "14",
    "อ่างทอง": "15",
    "ลพบุรี": "16",
    "สิงห์บุรี": "17",
    "ชัยนาท": "18",
    "สระบุรี": "19",
    "ชลบุรี": "20",
    "ระยอง": "21",
    "จันทบุรี": "22",
    "ตราด": "23",
    "ฉะเชิงเทรา": "24",
    "ปราจีนบุรี": "25",
    "นครนายก": "26",
    "สระแก้ว": "27",
    "นครราชสีมา": "30",
    "บุรีรัมย์": "31",
    "สุรินทร์": "32",
    "ศรีสะเกษ": "33",
    "อุบลราชธานี": "34",
    "ยโสธร": "35",
    "ชัยภูมิ": "36",
    "อำนาจเจริญ": "37",
    "บึงกาฬ": "38",
    "หนองบัวลำภู": "39",
    "ขอนแก่น": "40",
    "อุดรธานี": "41",
    "เลย": "42",
    "หนองคาย": "43",
    "มหาสารคาม": "44",
    "ร้อยเอ็ด": "45",
    "กาฬสินธุ์": "46",
    "สกลนคร": "47",
    "นครพนม": "48",
    "มุกดาหาร": "49",
    "เชียงใหม่": "50",
    "ลำพูน": "51",
    "ลำปาง": "52",
    "อุตรดิตถ์": "53",
    "แพร่": "54",
    "น่าน": "55",
    "พะเยา": "56",
    "เชียงราย": "57",
    "แม่ฮ่องสอน": "58",
    "นครสวรรค์": "60",
    "อุทัยธานี": "61",
    "กำแพงเพชร": "62",
    "ตาก": "63",
    "สุโขทัย": "64",
    "พิษณุโลก": "65",
    "พิจิตร": "66",
    "เพชรบูรณ์": "67",
    "ราชบุรี": "70",
    "กาญจนบุรี": "71",
    "สุพรรณบุรี": "72",
    "นครปฐม": "73",
    "สมุทรสาคร": "74",
    "สมุทรสงคราม": "75",
    "เพชรบุรี": "76",
    "ประจวบคีรีขันธ์": "77",
    "นครศรีธรรมราช": "80",
    "กระบี่": "81",
    "พังงา": "82",
    "ภูเก็ต": "83",
    "สุราษฎร์ธานี": "84",
    "ระนอง": "85",
    "ชุมพร": "86",
    "สงขลา": "90",
    "สตูล": "91",
    "ตรัง": "92",
    "พัทลุง": "93",
    "ปัตตานี": "94",
    "ยะลา": "95",
    "นราธิวาส": "96",
}

PROVINCES = list(PROVINCE_ID_MAPPING.items())
len(PROVINCES)


77

In [8]:
PROVINCE_TO_COUNTY = {
    "เชียงใหม่": 1, "แม่ฮ่องสอน": 1, "ลำปาง": 1, "ลำพูน": 1,
    "เชียงราย": 1, "น่าน": 1, "พะเยา": 1, "แพร่": 1,
    "ตาก": 2, "พิษณุโลก": 2, "เพชรบูรณ์": 2, "สุโขทัย": 2, "อุตรดิตถ์": 2,
    "กำแพงเพชร": 3, "ชัยนาท": 3, "นครสวรรค์": 3, "พิจิตร": 3, "อุทัยธานี": 3,
    "นนทบุรี": 4, "ปทุมธานี": 4, "พระนครศรีอยุธยา": 4, "ลพบุรี": 4,
    "สระบุรี": 4, "สิงห์บุรี": 4, "อ่างทอง": 4, "นครนายก": 4,
    "กาญจนบุรี": 5, "นครปฐม": 5, "ประจวบคีรีขันธ์": 5, "เพชรบุรี": 5,
    "ราชบุรี": 5, "สมุทรสงคราม": 5, "สมุทรสาคร": 5, "สุพรรณบุรี": 5,
    "จันทบุรี": 6, "ฉะเชิงเทรา": 6, "ชลบุรี": 6, "ตราด": 6,
    "ปราจีนบุรี": 6, "ระยอง": 6, "สมุทรปราการ": 6, "สระแก้ว": 6,
    "กาฬสินธุ์": 7, "ขอนแก่น": 7, "มหาสารคาม": 7, "ร้อยเอ็ด": 7,
    "บึงกาฬ": 8, "เลย": 8, "นครพนม": 8, "หนองคาย": 8,
    "หนองบัวลำภู": 8, "อุดรธานี": 8, "สกลนคร": 8,
    "บุรีรัมย์": 9, "ชัยภูมิ": 9, "นครราชสีมา": 9, "สุรินทร์": 9,
    "อำนาจเจริญ": 10, "อุบลราชธานี": 10, "ศรีสะเกษ": 10,
    "ยโสธร": 10, "มุกดาหาร": 10,
    "กระบี่": 11, "ชุมพร": 11, "นครศรีธรรมราช": 11, "พังงา": 11,
    "ภูเก็ต": 11, "ระนอง": 11, "สุราษฎร์ธานี": 11,
    "ตรัง": 12, "นราธิวาส": 12, "ปัตตานี": 12, "พัทลุง": 12,
    "ยะลา": 12, "สงขลา": 12, "สตูล": 12,
    "กรุงเทพมหานคร": 13,
}

TYPE_NAME_MAP = {
    "J442": "Acute asthma",
    "J45": "Acute asthma",
    "I21": "Acute ischemic heart diseases",
    "I24": "Acute ischemic heart diseases",
    "I22": "Acute ischemic heart diseases",
    "J44": "Chronic obstructive pulmonary disease",
    "H10": "กลุ่มโรคตาอักเสบ",
    "L309": "กลุ่มโรคผิวหนังอักเสบ",
    "L50": "กลุ่มโรคผิวหนังอักเสบ",
}

DIAG_MAIN_MAPPING = [
    (2, 1, "Chronic obstructive pulmonary disease (J44)", "J44"),
    (4, 2, "Acute asthma (J45)", "J45"),
    (8, 4, "Acute ischemic heart diseases (I21)", "I21"),
    (16, 6, "Subsequent ST elevation (STEMI) and non-ST elevation (NSTEMI) myocardial infarction (I22)", "I22"),
    (32, 7, "Conjunctivitis (H10)", "H10"),
    (64, 8, "Eczema (L30.9)", "L309"),
    (128, 9, "Urticaria (L50)", "L50"),
    (2048, 3, "Acute asthma (J44.2)", "J442"),
    (4096, 5, "Acute ischemic heart diseases (I24)", "I24"),
]

MEASURE_SUFFIX_MAPPING = [
    ("m", "การวินิจฉัยโรคทั้งหมด"),
    ("z", "การวินิจฉัยโรคหลัก ร่วมกับ Z58.1"),
    ("y", "การวินิจฉัยโรคหลัก ร่วมกับ Y97"),
    ("zy", "การวินิจฉัยโรคหลัก ร่วมกับ Z58.1+Y97"),
]


def norm_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_api_response(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("data", "result", "rows", "items"):
            value = payload.get(key)
            if isinstance(value, list):
                return value
        return [payload]
    return []


def fetch_province(session, province_name, province_id, year=YEAR):
    body = {
        "tableName": TABLE_NAME,
        "year": str(year),
        "province": str(province_id),
        "type": "json",
    }

    last_error = None
    for attempt in range(1, RETRY_COUNT + 2):
        try:
            response = session.post(API_URL, json=body, timeout=REQUEST_TIMEOUT_SEC)
            response.raise_for_status()
            records = normalize_api_response(response.json())
            return pd.DataFrame(records), None
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt <= RETRY_COUNT:
                time.sleep(min(30, attempt * 2))

    return pd.DataFrame(), last_error


def to_numeric_series(series):
    return pd.to_numeric(series, errors="coerce").fillna(0)


def thai_year_to_ad(value):
    year = int(float(str(value).strip()))
    return year - 543 if year > 2400 else year


def week_to_month(year, week):
    try:
        return dt.date.fromisocalendar(int(year), int(week), 4).month
    except Exception:
        return 12 if int(week) == 53 else pd.NA


def transform_api_to_long_shape(api_df, province_name, province_id, year_thai=YEAR):
    if api_df.empty:
        api_df = pd.DataFrame(columns=["diag_main"])

    api_df = api_df.copy()
    api_df["diag_main"] = pd.to_numeric(api_df.get("diag_main"), errors="coerce")

    province_name = norm_text(province_name)
    province_code = int(province_id)
    year = thai_year_to_ad(year_thai)
    county = PROVINCE_TO_COUNTY.get(province_name)
    rows = []

    for diag_main, typediag_id, typediag, icd10 in DIAG_MAIN_MAPPING:
        diag_df = api_df[api_df["diag_main"] == diag_main]
        typediag_name = TYPE_NAME_MAP.get(icd10)

        for suffix, diagnosis in MEASURE_SUFFIX_MAPPING:
            for week in range(1, 54):
                api_col = f"w_{week:02d}_{suffix}"
                case = int(to_numeric_series(diag_df[api_col]).sum()) if api_col in diag_df.columns else 0
                rows.append({
                    "province_code": province_code,
                    "province_name": province_name,
                    "county": county,
                    "year": year,
                    "week": week,
                    "month": week_to_month(year, week),
                    "typediag_id": typediag_id,
                    "typediag": typediag,
                    "icd10": icd10,
                    "Typediag_name": typediag_name,
                    "diagnosis": diagnosis,
                    "case": case,
                })

    return pd.DataFrame(rows)


def finalize_long_df(df):
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)

    final_df = df.copy()
    final_df["province_name"] = final_df["province_name"].astype(str).map(norm_text)
    final_df["typediag"] = final_df["typediag"].astype(str).map(norm_text)
    final_df["diagnosis"] = final_df["diagnosis"].astype(str).map(norm_text)

    final_df = final_df.sort_values(
        by=["province_code", "year", "typediag_id", "typediag", "diagnosis", "week"],
        ascending=[True, True, True, True, True, True],
    ).reset_index(drop=True)

    final_df.insert(0, "no", range(1, len(final_df) + 1))
    return final_df[FINAL_COLUMNS].copy()


def write_csv_atomic(df, path):
    tmp_path = path.with_name(f".{path.name}.tmp")
    try:
        df.to_csv(tmp_path, index=False, encoding="utf-8-sig")
        os.replace(tmp_path, path)
    except Exception:
        tmp_path.unlink(missing_ok=True)
        raise


In [9]:
all_frames = []
summary_rows = []

with requests.Session() as session:
    session.headers.update({"Content-Type": "application/json"})

    for index, (province_name, province_id) in enumerate(PROVINCES, start=1):
        print(f"[{index:02d}/{len(PROVINCES)}] {province_name} ({province_id})")
        api_df, error = fetch_province(session, province_name, province_id)
        long_df = transform_api_to_long_shape(api_df, province_name, province_id)

        if error is None:
            all_frames.append(long_df)
            status = "SUCCESS"
            detail = "OK"
        else:
            status = "FAILED"
            detail = error

        summary_rows.append({
            "year": YEAR,
            "province": province_name,
            "provinceId": province_id,
            "status": status,
            "api_rows": len(api_df),
            "export_rows": len(long_df) if error is None else 0,
            "export_columns": len(long_df.columns),
            "detail": detail,
        })
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

hdc_df = finalize_long_df(pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame())
summary_df = pd.DataFrame(summary_rows)

print("HDC shape:", hdc_df.shape)
hdc_df.head()


[01/77] กรุงเทพมหานคร (10)
[02/77] สมุทรปราการ (11)
[03/77] นนทบุรี (12)
[04/77] ปทุมธานี (13)
[05/77] พระนครศรีอยุธยา (14)
[06/77] อ่างทอง (15)
[07/77] ลพบุรี (16)
[08/77] สิงห์บุรี (17)
[09/77] ชัยนาท (18)
[10/77] สระบุรี (19)
[11/77] ชลบุรี (20)
[12/77] ระยอง (21)
[13/77] จันทบุรี (22)
[14/77] ตราด (23)
[15/77] ฉะเชิงเทรา (24)
[16/77] ปราจีนบุรี (25)
[17/77] นครนายก (26)
[18/77] สระแก้ว (27)
[19/77] นครราชสีมา (30)
[20/77] บุรีรัมย์ (31)
[21/77] สุรินทร์ (32)
[22/77] ศรีสะเกษ (33)
[23/77] อุบลราชธานี (34)
[24/77] ยโสธร (35)
[25/77] ชัยภูมิ (36)
[26/77] อำนาจเจริญ (37)
[27/77] บึงกาฬ (38)
[28/77] หนองบัวลำภู (39)
[29/77] ขอนแก่น (40)
[30/77] อุดรธานี (41)
[31/77] เลย (42)
[32/77] หนองคาย (43)
[33/77] มหาสารคาม (44)
[34/77] ร้อยเอ็ด (45)
[35/77] กาฬสินธุ์ (46)
[36/77] สกลนคร (47)
[37/77] นครพนม (48)
[38/77] มุกดาหาร (49)
[39/77] เชียงใหม่ (50)
[40/77] ลำพูน (51)
[41/77] ลำปาง (52)
[42/77] อุตรดิตถ์ (53)
[43/77] แพร่ (54)
[44/77] น่าน (55)
[45/77] พะเยา (56)
[46/77] เชียงราย (57)
[47/7

,no,province_code,province_name,county,year,week,month,typediag_id,typediag,icd10,Typediag_name,diagnosis,case
0,1,10,กรุงเทพมหานคร,13,2026,1,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,12
1,2,10,กรุงเทพมหานคร,13,2026,2,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,29
2,3,10,กรุงเทพมหานคร,13,2026,3,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,34
3,4,10,กรุงเทพมหานคร,13,2026,4,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,44
4,5,10,กรุงเทพมหานคร,13,2026,5,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,28


In [10]:
write_csv_atomic(hdc_df, LONG_OUTPUT_FILE)
write_csv_atomic(summary_df, SUMMARY_OUTPUT_FILE)

success_count = int(summary_df["status"].eq("SUCCESS").sum()) if not summary_df.empty else 0
failed_count = int(summary_df["status"].eq("FAILED").sum()) if not summary_df.empty else 0
total_api_rows = int(summary_df["api_rows"].sum()) if not summary_df.empty else 0
total_export_rows = int(summary_df["export_rows"].sum()) if not summary_df.empty else 0
long_size_mb = LONG_OUTPUT_FILE.stat().st_size / 1024 / 1024 if LONG_OUTPUT_FILE.exists() else 0
summary_size_kb = SUMMARY_OUTPUT_FILE.stat().st_size / 1024 if SUMMARY_OUTPUT_FILE.exists() else 0

print("Export completed")
print("- Long file:", LONG_OUTPUT_FILE.resolve())
print("- Summary file:", SUMMARY_OUTPUT_FILE.resolve())
print(f"- Provinces: success={success_count}, failed={failed_count}, total={len(summary_df)}")
print(f"- API rows: {total_api_rows:,}")
print(f"- Export rows: {total_export_rows:,}")
print(f"- Long size: {long_size_mb:.2f} MB")
print(f"- Summary size: {summary_size_kb:.1f} KB")

if FAIL_ON_PROVINCE_ERROR and failed_count:
    failed_items = "; ".join(
        f"{row.province}({row.provinceId}): {row.detail}"
        for row in summary_df[summary_df["status"].eq("FAILED")].itertuples()
    )
    raise RuntimeError(f"HDC API failed for {failed_count} province(s): {failed_items}")

hdc_df.head(20)


Export completed
- Long file: /Users/champ/Documents/dev/envocc-dashboard-pm/airflow/dags/scripts/hdc_merged_long_2569.csv
- Summary file: /Users/champ/Documents/dev/envocc-dashboard-pm/airflow/dags/scripts/hdc_report_summary_2569.csv
- Provinces: success=77, failed=0, total=77
- API rows: 36,344
- Export rows: 146,916
- Long size: 28.36 MB
- Summary size: 4.2 KB


,no,province_code,province_name,county,year,week,month,typediag_id,typediag,icd10,Typediag_name,diagnosis,case
0,1,10,กรุงเทพมหานคร,13,2026,1,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,12
1,2,10,กรุงเทพมหานคร,13,2026,2,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,29
2,3,10,กรุงเทพมหานคร,13,2026,3,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,34
3,4,10,กรุงเทพมหานคร,13,2026,4,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,44
4,5,10,กรุงเทพมหานคร,13,2026,5,1,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,28
5,6,10,กรุงเทพมหานคร,13,2026,6,2,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,90
6,7,10,กรุงเทพมหานคร,13,2026,7,2,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,89
7,8,10,กรุงเทพมหานคร,13,2026,8,2,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,89
8,9,10,กรุงเทพมหานคร,13,2026,9,2,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,77
9,10,10,กรุงเทพมหานคร,13,2026,10,3,1,Chronic obstructive pulmonary disease (J44),J44,Chronic obstructive pulmonary disease,การวินิจฉัยโรคทั้งหมด,91
